# 01 — Data Exploration

Explore and characterise a fine-tuning dataset before training:
- Token length distribution
- Quality filter pass rates
- Conversation structure (for chat datasets)
- Vocabulary coverage

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import AutoTokenizer

from src.data.preprocessing import PreprocessingConfig, TextPreprocessor
from src.data.tokenisation import analyse_lengths

sns.set_theme(style='darkgrid', palette='husl')
plt.rcParams['figure.dpi'] = 120

In [ ]:
# Load dataset
DATASET = 'HuggingFaceH4/ultrachat_200k'
SPLIT = 'train_sft'
N_SAMPLES = 5000

ds = load_dataset(DATASET, split=SPLIT).shuffle(seed=42).select(range(N_SAMPLES))
print(f'Loaded {len(ds)} samples')
print(f'Columns: {ds.column_names}')
ds[0]

In [ ]:
# Token length analysis
MODEL = 'meta-llama/Meta-Llama-3-8B'
tokenizer = AutoTokenizer.from_pretrained(MODEL)

# Flatten all messages to text for length analysis
texts = [
    tokenizer.apply_chat_template(ex['messages'], tokenize=False)
    for ex in ds
]

stats = analyse_lengths(texts, tokenizer, percentiles=[50, 75, 90, 95, 99, 100])
for k, v in stats.items():
    print(f'{k:25s}: {v}')

In [ ]:
# Token length histogram
lengths = [len(tokenizer(t, add_special_tokens=True)['input_ids']) for t in texts]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(lengths, bins=60, color='steelblue', alpha=0.8)
axes[0].axvline(stats['p95'], color='red', linestyle='--', label=f'p95={stats["p95"]}')
axes[0].axvline(2048, color='orange', linestyle='--', label='max_seq=2048')
axes[0].set_xlabel('Sequence Length (tokens)')
axes[0].set_ylabel('Count')
axes[0].set_title('Token Length Distribution')
axes[0].legend()

axes[1].hist(lengths, bins=60, color='steelblue', alpha=0.8, cumulative=True, density=True)
axes[1].axhline(0.95, color='red', linestyle='--', label='95th pct')
axes[1].set_xlabel('Sequence Length (tokens)')
axes[1].set_ylabel('Cumulative Fraction')
axes[1].set_title('Cumulative Length Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig('length_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# Quality filter analysis
preprocessor = TextPreprocessor(PreprocessingConfig())
pass_count = sum(1 for t in texts if preprocessor._passes_quality(t))
print(f'Quality filter pass rate: {pass_count}/{len(texts)} ({100*pass_count/len(texts):.1f}%)')

In [ ]:
# Conversation structure analysis
turn_counts = [len([m for m in ex['messages'] if m['role'] == 'user']) for ex in ds]

plt.figure(figsize=(8, 4))
plt.hist(turn_counts, bins=range(1, max(turn_counts)+2), align='left', color='steelblue', alpha=0.8)
plt.xlabel('Number of User Turns')
plt.ylabel('Count')
plt.title('Conversation Turn Distribution')
plt.xticks(range(1, min(max(turn_counts)+1, 20)))
plt.tight_layout()
plt.show()

print(f'Mean turns: {sum(turn_counts)/len(turn_counts):.1f}')
print(f'Max turns: {max(turn_counts)}')